# Контест 1 [11 баллов]
## Вариант 2

#### Импорты и настройка

В этой ячейке собраны все необходимые импорты.

**Ничего менять не нужно — просто запусти ячейку.**

In [231]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random  # добавил импорт сам

w = 30

## Классическая вероятность. Условная вероятность [6 баллов]


**Контекст**

В ИТ-архитектуре крупного маркетплейса запросы к базе данных обрабатываются основным сервером — назовем его нода А. Если нода А перегружена и выдает ошибку, система мгновенно перенаправляет этот запрос одновременно на две резервные ноды — ноду Б и ноду В.

Запрос считается успешно обработанным, если:
- на него ответила нода А

либо

- нода А не ответила, но ответила хотя бы одна из резервных нод.

Если нода А не ответила и обе резервные ноды тоже выдали ошибку, транзакция падает с ошибкой 500  (internal server error).

Вероятности отказов:
- $p_a = 0.4$ — для ноды А,
- $p_b = 0.2$ — для ноды Б,
- $p_c = 0.3$ — для ноды В.

Отказы нод считаются независимыми событиями.

*Указание.* В задачах ниже используй подготовленные шаблоны функций. При необходимости ты также можешь создавать свои функции.

In [232]:
FAILURE_PROBABILITIES = (0.4, 0.2, 0.3)


### Задача 1 [2 балла]

Напиши алгоритм расчета точной вероятности того, что запрос будет успешно обработан.

In [233]:
def calculate_exact_probability(
        p1: float,
        p2: float,
        p3: float
) -> float:
    """Вычисляет вероятность успешной обработки запроса.
    считаю что p1 это мастер нода
    """

    # нода 1 ответила ИЛИ при условии что нода 1 не ответила ответила нода 2 ИЛИ при условии что нода 1 не ответила ответила нода 3
    return 1 - p1 + (1 - p1) * ((p1 * (1 - p2)) / (p1 + (1 - p2)) + (p1 * (1 - p3)) / (p1 + (1 - p3))) + (
            p1 * (1 - p2)) / (p1 + (1 - p2)) * (p1 * (1 - p3)) / (p1 + (1 - p3))



In [234]:
prob = 0.0

##### Твой код здесь
prob = calculate_exact_probability(*FAILURE_PROBABILITIES)

print(f"  {'Точная вероятность успешной обработки запроса:':<{w}} {prob}")

  Точная вероятность успешной обработки запроса: 0.9806060606060606


### Задача 2 [2 балла]

Напиши алгоритм **приближенного** вычисления вероятности успешной транзакции с помощью метода Монте-Карло. Сравни результат с полученной на предыдущем этапе точной вероятностью.

*Указание*. Для генерации случайных чисел используй библиотеку `random`.

In [235]:
def trace_request(
        p1: float,
        p2: float,
        p3: float
) -> bool:
    if random.random() > p1:
        return True
    if random.random() > p2 or random.random() > p3:
        return True
    return False


def monte_carlo(
        p1: float,
        p2: float,
        p3: float,
        iterations: int = 1_000_000
) -> float:
    """Оценивает вероятность события методом Монте-Карло."""

    success_cnt = 0
    experiment_cnt = 0
    for _ in range(iterations):
        experiment_cnt += 1
        success_cnt += trace_request(p1, p2, p3)

    return success_cnt / experiment_cnt

In [236]:
prob_mc = 0.0

prob_mc = monte_carlo(*FAILURE_PROBABILITIES)


In [237]:
##### Вывод результатов на экран. Ничего менять здесь не нужно
print(f"━━━ Подсчет вероятности успешной обработки запроса ━━━")
print(f"  {'Вероятность Монте-Карло:':<{w}} {prob_mc:.6f} (примерно {prob_mc * 100:.3f}%)")
print(f"  {'Точная вероятность':<{w}} {prob:.6f} (примерно {prob * 100:.3f}%)")
print(f"  {'Абсолютная разность':<{w}} {prob - prob_mc:.6f}")
print(f"  {'Относительная разность':<{w}} {(prob - prob_mc) / prob:.4f}%")

━━━ Подсчет вероятности успешной обработки запроса ━━━
  Вероятность Монте-Карло:       0.976272 (примерно 97.627%)
  Точная вероятность             0.980606 (примерно 98.061%)
  Абсолютная разность            0.004334
  Относительная разность         0.0044%


### Задача 3 [1 балл]

Пусть известно, что запрос был успешно выполнен. Напиши алгоритм, вычисляющий точную вероятность того, что в процессе обработки запроса нода А выдала ошибку.

In [238]:
def calculate_exact_probability_conditional(
        p1: float,
        p2: float,
        p3: float
) -> float:
    """Вычисляет вероятность того, что в процессе обработки запроса нода А выдала ошибку.
    """

    return p1

In [239]:
prob_cond = calculate_exact_probability_conditional(*FAILURE_PROBABILITIES)

print(f"  {'Точная вероятность того, что в процессе обработки запроса нода А выдала ошибку:':<{w}} {prob_cond}")

  Точная вероятность того, что в процессе обработки запроса нода А выдала ошибку: 0.4


### Задача 4 [1 балл]

Напиши выводы по этому блоку.

1. Метод монте карло получил хорошее приближение к настоящему ответу (0.0048% отклонения)
2. Монте карло работает дольше чем аналитическое решение (267ms vs 50ms)
3. По этому мы можем использовать монте карло когда тяжело сформулировать вероятность сразу, но легко описать эксперимент и мы можем потратить немного больше времени

## Геометрическая вероятность [5 баллов]

**Контекст**

Бизнес-аналитики маркетплейса исследуют профили новых продавцов по двум стандартизированным метрикам:
- $x$ — индекс эффективности рекламных расходов,
- $y$ — индекс качества обслуживания клиентов.

Области значений метрик: $x \in [-10; 10], y \in [-10; 10]$.

Модель относит продавца к категории «аномальный профиль», если выполнено три условия:
- $x > 0$,
- $y < 0$,
- $|{}x-y|{}\le 6$.


### Задача 1 [2 балла]

Метрики нового продавца распределены равномерно в квадрате $[-10;10] \times [-10;10]$. Найди точную вероятность того, что продавец попадет в категорию «аномальный профиль».

In [240]:
def geom_probability(
) -> float:
    """Вычисляет точную геометрическую вероятность.
    """

    return (5 * 5 - 2 * 2) / (10 * 10)

In [241]:
prob_geom = geom_probability()

print(f"  {'Геометрическая вероятность попадания точки в обозначенную зону:':<{w}} {prob_geom}")

  Геометрическая вероятность попадания точки в обозначенную зону: 0.21


### Задача 2 [2 балла]

Напиши алгоритм приближенного вычисления такой вероятности с помощью метода Монте-Карло и сравни результат с полученной на предыдущем этапе точной вероятностью. Для генерации случайных чисел используй библиотеку `random`.

In [242]:
def experiment() -> bool:
    x, y = random.uniform(-10, 10), random.uniform(-10, 10)
    return x > 0 and y > 0 and abs(x - y) <= 6


def monte_carlo_geom(
        iterations: int = 1_000_000,
) -> float:
    """Оценивает вероятность события методом Монте-Карло."""

    success_cnt = 0
    experiment_cnt = 0
    for _ in range(iterations):
        experiment_cnt += 1
        success_cnt += experiment()

    return success_cnt / experiment_cnt

In [243]:
prob_geom_mc = monte_carlo_geom()


In [244]:
##### Выводы. Менять ничего не нужно.
print(f"━━━ Симуляция Монте-Карло ━━━")
print(f"  {'Вероятность Монте-Карло:':<{w}} {prob_geom_mc:.6f} (примерно {prob_geom_mc * 100:.3f}%)")
print(f"  {'Точная вероятность':<{w}} {prob_geom:.6f} (примерно {prob_geom * 100:.3f}%)")
print(f"  {'Абсолютная разность':<{w}} {prob_geom - prob_geom_mc:.6f}")
print(f"  {'Относительная разность':<{w}} {(prob_geom - prob_geom_mc) / prob_geom:.4f}%")

━━━ Симуляция Монте-Карло ━━━
  Вероятность Монте-Карло:       0.209869 (примерно 20.987%)
  Точная вероятность             0.210000 (примерно 21.000%)
  Абсолютная разность            0.000131
  Относительная разность         0.0006%


### Задача 3 [1 балл]

Напиши выводы по этому блоку.

1. Метод монте карло получил хорошее приближение к настоящему ответу (0.0009% отклонения)
2. Монте карло работает дольше чем аналитическое решение (635ms vs 50ms)
3. По этому мы можем использовать монте карло когда тяжело сформулировать вероятность сразу, но легко описать эксперимент и мы можем потратить немного больше времени


после написания ноутбука все еще раз прогнал, так что точные цифры в выводах могут отличаться